# Module 5: Agent Frameworks
# Topic 32: LCEL (LangChain Expression Language)

> **Interview Difficulty:** ⭐⭐⭐⭐⭐ (Must Know)
>
> **Interview Frequency:** Extremely High
>
> **Prerequisites:**
> - Models ✅
> - Prompt Templates ✅
> - Messages ✅
> - Output Parsers ✅

---

# Learning Objectives

After this topic, you should be able to answer:

- What is LCEL?
- Why was LCEL introduced?
- What is Runnable?
- How does the `|` operator work?
- LCEL Execution Flow
- invoke(), batch(), stream(), ainvoke()
- Sequential Composition
- Parallel Execution
- Branching
- Best Practices
- Interview Questions

---

# 1. What is LCEL?

LCEL stands for **LangChain Expression Language**.

It is a declarative way of connecting LangChain components together.

Instead of manually calling each component one after another, LCEL lets you compose them into a pipeline.

---

# Interview Definition ⭐⭐⭐⭐⭐

> **LCEL (LangChain Expression Language) is a declarative pipeline syntax that composes LangChain components such as prompts, models, output parsers, and custom logic into reusable execution workflows using Runnable objects.**

---

# 2. Why Was LCEL Introduced?

Before LCEL, developers manually orchestrated every step.

Example:

```python
formatted_prompt = prompt.invoke(data)

response = llm.invoke(formatted_prompt)

result = parser.invoke(response)
```

As applications grew larger:

- More boilerplate
- Harder debugging
- Difficult composition
- Less reusable code

---

LCEL simplifies this.

```python
chain = prompt | llm | parser
```

One line replaces multiple manual steps.

---

# 3. Without LCEL

```text
User Input
      │
      ▼
Prompt Template
      │
      ▼
invoke()
      │
      ▼
Chat Model
      │
      ▼
invoke()
      │
      ▼
Output Parser
      │
      ▼
Application
```

Manual execution.

---

# 4. With LCEL

```text
User Input

↓

Prompt

↓

LLM

↓

Parser

↓

Application
```

Everything becomes one reusable pipeline.

---

# 5. What is a Runnable?

This is the most important concept behind LCEL.

A **Runnable** is any LangChain component that can execute work.

Examples:

- PromptTemplate
- ChatPromptTemplate
- ChatModel
- OutputParser
- Lambda Function
- Custom Runnable

Almost everything in modern LangChain is a Runnable.

---

# Interview Definition

> **A Runnable is any object in LangChain that supports standardized execution methods such as invoke(), batch(), stream(), or ainvoke().**

---

# 6. LCEL Architecture

```text
              Runnable

                  │

    ┌─────────────┼─────────────┐

    ▼             ▼             ▼

 Prompt        Chat Model    Parser

                  │

                  ▼

             Final Output
```

Every block is a Runnable.

---

# 7. The Pipe (`|`) Operator

Probably the most asked LangChain interview question.

Example

```python
chain = prompt | llm
```

What happens internally?

```
Output of Prompt

↓

Input of LLM
```

The output of the left Runnable becomes the input of the right Runnable.

---

Another example

```python
chain = prompt | llm | parser
```

Execution

```text
Prompt

↓

LLM

↓

Parser

↓

Result
```

---

# 8. First LCEL Example

```python
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

prompt = ChatPromptTemplate.from_template(

    "Explain {topic}"

)

llm = ChatOpenAI(

    model="gpt-4o-mini"

)

chain = prompt | llm

response = chain.invoke(

    {

        "topic":"LangChain"

    }

)

print(response.content)
```

---

# 9. What Happens Internally?

Instead of

```python
formatted = prompt.invoke(data)

response = llm.invoke(formatted)
```

LCEL automatically does

```python
Prompt

↓

Formatted Prompt

↓

LLM

↓

AIMessage
```

You only call

```python
chain.invoke(...)
```

---

# 10. Adding an Output Parser

Without parser

```python
prompt | llm
```

Output

```
AIMessage
```

With parser

```python
prompt | llm | parser
```

Output

```
Python Object

or

JSON

or

String
```

---

# 11. Complete Pipeline

```python
chain = (

    prompt

    | llm

    | parser

)
```

Execution

```text
Variables

↓

Prompt

↓

Messages

↓

LLM

↓

AIMessage

↓

Parser

↓

Python Object
```

---

# 12. invoke()

Runs a single request.

```python
response = chain.invoke(

    {

        "topic":"AI"

    }

)
```

---

# 13. batch()

Runs multiple requests.

```python
responses = chain.batch(

    [

        {"topic":"AI"},

        {"topic":"ML"},

        {"topic":"NLP"}

    ]

)
```

Useful for bulk inference.

---

# 14. stream()

Streams tokens.

```python
for chunk in chain.stream(

    {

        "topic":"AI"

    }

):

    print(chunk.content,end="")
```

---

# 15. ainvoke()

Runs asynchronously.

```python
response = await chain.ainvoke(

    {

        "topic":"AI"

    }

)
```

Ideal for FastAPI and async applications.

---

# 16. RunnableLambda

You can insert your own Python logic.

Example

```python
from langchain_core.runnables import RunnableLambda

def to_upper(text):

    return text.upper()

uppercase = RunnableLambda(to_upper)
```

Pipeline

```python
prompt

|

llm

|

uppercase
```

---

# 17. RunnableParallel

Run multiple tasks simultaneously.

Example

```text
User Question

        │

        ▼

RunnableParallel

   │           │

   ▼           ▼

Summary    Keywords

   │           │

   └─────┬─────┘

         ▼

Combined Output
```

Very useful for production systems.

---

# 18. RunnableBranch

Conditional execution.

Example

```text
Question

      │

      ▼

SQL?

 │         │

Yes       No

│          │

▼          ▼

SQL      LLM

```

Very useful in AI routing.

---

# 19. LCEL vs Traditional Code

| Traditional | LCEL |
|--------------|------|
| Manual invoke | Automatic |
| More boilerplate | Concise |
| Harder composition | Easy composition |
| Manual chaining | Pipeline syntax |

---

# 20. Advantages

✅ Readable

✅ Modular

✅ Reusable

✅ Supports streaming

✅ Supports async

✅ Supports parallel execution

✅ Supports branching

---

# 21. Common Mistakes

❌ Thinking `|` is a Python bitwise operator here.

It is **operator overloading** implemented by LangChain.

---

❌ Calling `invoke()` on every component manually.

Instead

```python
chain.invoke()
```

---

❌ Forgetting every component must be Runnable.

---

# 22. Real Enterprise Example

```python
chain = (

    prompt

    | retriever

    | llm

    | parser

)
```

Execution

```text
Question

↓

Prompt

↓

Retriever

↓

Relevant Documents

↓

LLM

↓

Parser

↓

JSON
```

Notice how every stage is simply connected using LCEL.

---

# 23. Complete Execution Flow

```text
User

↓

Variables

↓

Prompt

↓

Chat Model

↓

AIMessage

↓

Parser

↓

Application
```

---

# 24. Interview Questions

## Q1. What is LCEL?

**Answer:**

LCEL (LangChain Expression Language) is a declarative syntax for composing LangChain components into reusable execution pipelines using Runnable objects.

---

## Q2. What is a Runnable?

**Answer:**

A Runnable is any LangChain component that implements standardized execution methods such as invoke(), batch(), stream(), or ainvoke().

---

## Q3. What does the `|` operator do?

**Answer:**

It composes two Runnable objects. The output of the left Runnable automatically becomes the input of the right Runnable.

---

## Q4. What are the advantages of LCEL?

**Answer:**

- Less boilerplate
- Better readability
- Reusability
- Async support
- Streaming
- Parallel execution
- Branching

---

## Q5. What is RunnableParallel?

**Answer:**

RunnableParallel executes multiple Runnable components simultaneously and combines their outputs.

---

## Q6. What is RunnableBranch?

**Answer:**

RunnableBranch enables conditional execution by selecting different execution paths based on input conditions.

---

# 25. Quick Revision

| Component | Purpose |
|-----------|----------|
| Runnable | Executable component |
| invoke() | Single execution |
| batch() | Multiple executions |
| stream() | Token streaming |
| ainvoke() | Async execution |
| RunnableLambda | Custom Python logic |
| RunnableParallel | Parallel execution |
| RunnableBranch | Conditional routing |

---

# Interview Cheat Sheet

```text
User

↓

Variables

↓

Prompt

↓

Chat Model

↓

AIMessage

↓

Parser

↓

Application

Everything is a Runnable

Prompt

↓

LLM

↓

Parser

Connected using

|

Execution Methods

invoke()

batch()

stream()

ainvoke()
```

---

# 26. Frequently Asked Interview Coding Question

### Question

What does this line mean?

```python
chain = prompt | llm | parser
```

### Answer

- `prompt` formats the input.
- The formatted prompt is automatically passed to `llm`.
- The LLM generates an `AIMessage`.
- The parser converts the output into the required format.
- The developer only invokes the pipeline once using `chain.invoke()`.

---

# 30-Second Interview Answer

> **LCEL (LangChain Expression Language) is LangChain's pipeline syntax for composing Runnable components such as prompts, models, output parsers, and custom logic. Using the `|` operator, developers can build readable, reusable, and scalable execution pipelines that support synchronous, asynchronous, streaming, parallel, and conditional execution without manually orchestrating each step.**

---

# Key Takeaway

> **LCEL is the backbone of modern LangChain. Instead of manually connecting prompts, models, and parsers, developers compose Runnable objects into execution pipelines using the `|` operator, making AI applications cleaner, modular, and production-ready.**